# US Accidents Analysis


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


# SECTION 1: LOAD


In [ ]:
# Load the CSV file
df_us = pd.read_csv('data/raw/US_Accidents_23.csv')

print('DATASET LOADED')
print(f'Total Accidents: {len(df_us)}')
print(f'Total Columns: {df_us.shape[1]}')


# SECTION 2: EXPLORE


In [ ]:
print('FIRST 5 ROWS:')
df_us.head()

In [ ]:
# Show all columns
print('ALL COLUMNS:')
print(df_us.columns.tolist())


In [ ]:
# Data types and info
print('DATA TYPES:')
df_us.dtypes


In [ ]:
# calculate not null values for each column and the total not null values
print(f'\nThe not nulls values as per column:\n{df_us.count()}')
print(f'\nTotal not null values:\n{df_us.notnull().sum().sum()}')



In [ ]:
#missing values per column and total missing values
missing_values = df_us.isnull().sum()
print(f'\nMissing values per column:\n{missing_values}')
print(f'\nTotal missing values:\n{missing_values.sum()}')




In [ ]:
#percentage of missing values
missing_percentage = (missing_values / len(df_us)) * 100
print(f'\nPercentage of missing values per column:\n{missing_percentage.round()}')

In [ ]:
df_us.describe()

In [ ]:
# Categorical data 
print('Categorical data:\n')
for col in ['State', 'City', 'Severity','Weather_Condition']:
    print(f'{col} value counts: {df_us[col].nunique()}')



In [ ]:
# Top 10 state with the hightest number of accidents
print('Top 10 States:')
print(df_us['State'].value_counts().head(10))


In [ ]:
# Top 10 cities with the hightest number of accidents

print('Top 10 Cities:')
print(df_us['City'].value_counts().head(10))


In [ ]:
# The severity levels of accidents
print('Severity Levels:')
print(df_us['Severity'].value_counts().sort_index())



# SECTION 3: CLEAN THE DATA

In [ ]:
# Count missing values in each column
missing_count = df_us.isnull().sum()
print('Missing values per column:')
print(missing_count[missing_count > 0])
print(f'\nTotal missing: {missing_count.sum()}')

In [ ]:
# Drop unused columns
unused_cols = [
    'End_Lat', 'End_Lng','Wind_Chill(F)', 'Humidity(%)','ID', 'Source','Description', 'Street', 'County', 
    'Zipcode', 'Country', 'Timezone', 'Airport_Code','Weather_Timestamp','Wind_Direction',
    'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 
    'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Turning_Loop','Sunrise_Sunset', 
    'Civil_Twilight', 'Nautical_Twilight', 'Astronomical_Twilight' #Cols not used
]

before_cols = df_us.shape[1]

df_us = df_us.drop(columns=[col for col in unused_cols if col in df_us.columns])

print("Removed:", before_cols - df_us.shape[1])
print('the shape of data after dropping unused columns:', df_us.shape)

In [ ]:
# Fill missing values into numeric columns with their median value
numeric_cols = df_us.select_dtypes(include='number').columns
df_us[numeric_cols] = df_us[numeric_cols].fillna(df_us[numeric_cols].median())
print('null values in numeric columns are filled with median')


In [ ]:
# Categorical columns to fill
categorical_cols = ['Weather_Condition', 'City', 'State', 'Severity']

df_us[categorical_cols] = df_us[categorical_cols].fillna('Null')
print('categorical columns are filled with Null')


In [ ]:
# Check for remaining missing values
total_missing = df_us.isnull().sum().sum()

if total_missing == 0:
    print('The missing values filled\n')
else:
    print(f'{total_missing} of missing values remaining\n')

print(f'dataset shape: {df_us.shape}')


In [ ]:
# Remove duplicates
print('Removing duplicates:')
before = len(df_us)
print(f'Rows before remove duplicates: {before}')
# Remove duplicates
df_us = df_us.drop_duplicates()
after = len(df_us)
print(f'Duplicates removed: {before - after}')


In [ ]:
print(f'\ndataset shape: {df_us.shape}')


In [ ]:

# Convert time columns to datetime
df_us['Start_Time'] = pd.to_datetime(df_us['Start_Time'], errors='coerce')
df_us['End_Time'] = pd.to_datetime(df_us['End_Time'], errors='coerce')

# Calculate duration in minutes
df_us['Duration_Minutes'] = (df_us['End_Time'] - df_us['Start_Time']).dt.total_seconds() / 60
# Remove rows with long durations > 24 hours (Outlairs)
before = len(df_us)
df_us = df_us[df_us['Duration_Minutes'] <= 1440]

print('removed:', before - len(df_us))
print('length of dataset:', len(df_us))


In [ ]:
# Verify cleaning
print(f'Missing values: {df_us.isnull().sum().sum()}')
print(f'\nCleaned dataset shape: {df_us.shape}')

In [ ]:
# Step 1: Save cleaned data to CSV
df_us.to_csv('us_accidents_cleaned.csv', index=False)
print('Cleaned data saved into ---> us_accidents_cleaned.csv')


In [ ]:
#Load the cleaned dataset
df_us = pd.read_csv('us_accidents_cleaned.csv')
print('data loaded!')
print(f'Dataset shape: {df_us.shape}')


In [ ]:

df_us.head(5)


# SECTION 4: TRANSFORM 


In [ ]:
df_us = df_us.dropna(subset=['Start_Time'])

df_us['Start_Time'] = pd.to_datetime(df_us['Start_Time'])

start = df_us['Start_Time'].min()
end = df_us['Start_Time'].max()

print('From:', start)
print('To:', end)
print('Duration (days):', (end - start).days)

In [ ]:
# accident duration
df_us['Start_Time'] = pd.to_datetime(df_us['Start_Time'], errors='coerce')
df_us['End_Time'] = pd.to_datetime(df_us['End_Time'], errors='coerce')

df_us['Duration_Minutes'] = (df_us['End_Time'] - df_us['Start_Time']).dt.total_seconds() / 60
print('Duration state:')
print(f'{df_us['Duration_Minutes'].describe().round(2)}')

max_duration = df_us['Duration_Minutes'].max()
max_hours = max_duration / 60

print(f'\nLongest accident: {max_duration} minutes')


# SECTION 5: ANALYZE 

In [ ]:
# dataset overview
start = df_us['Start_Time'].min()
end = df_us['Start_Time'].max()

print(f"Total accidents: {len(df_us)}")
print(f"From {start.strftime('%dth of %B %Y')} - to {end.strftime('%dth of %B %Y')}")
print(f"States: {df_us['State'].nunique()}")
print(f"Cities: {df_us['City'].nunique()}")

In [ ]:
# Calculate severity of accidents (Value counts and percentages)
sev = df_us['Severity'].value_counts().sort_index()
sev_pct = (sev / len(df_us) * 100)
print('Severity levels:')
for level in sev.index:
    print(f'Level {level}: {sev[level]} ({sev_pct[level]:.2f}%)')

In [ ]:
print(f'Locations of accidents:\n')
print("Top 10 States:")
for state, count in df_us['State'].value_counts().head(10).items():
    print(f"{state}: {count:,} ({count/len(df_us)*100:.2f}%)")

print("\nTop 10 Cities:")
for city, count in df_us['City'].value_counts().head(10).items():
    print(f"{city}: {count:,} ({count/len(df_us)*100:.2f}%)")


In [ ]:
print('Highest times (Hours - Day - Month):\n')

df_us['Hour'] = df_us['Start_Time'].dt.hour
df_us['DayOfWeek'] = df_us['Start_Time'].dt.day_name()
df_us['Month'] = df_us['Start_Time'].dt.month

# hours(3 highest hours)
hours = df_us['Hour'].value_counts().head(3).sort_index(ascending=False)
print("Highest 3 Hours:")
for h, c in hours.items():
    print(f"{h}:00 ---> {c:,} accidents")

# days (highest day)
days_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_counts = df_us['DayOfWeek'].value_counts().reindex(days_order)
day = day_counts.idxmax()
print(f"\nHighest Day: {day} ({day_counts[day]} accidents)\n")

# months (highest months)
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
month_counts = df_us['Month'].value_counts().sort_index()
month = month_counts.idxmax()
print(f"Highest Month: {months[month-1]} ({month_counts[month]} accidents)")

In [ ]:
print('Duration of accidentes statistics:')
print(f'Average: {df_us["Duration_Minutes"].mean():.1f} min')
print(f'Median: {df_us["Duration_Minutes"].median():.1f} min')
print(f'Max: {df_us["Duration_Minutes"].max():.0f} min')

In [ ]:
print("Top 5 weather conditions:\n")
weather = df_us['Weather_Condition'].value_counts().head(5)
for w, i in weather.items():
    print(f"{w}: {i} ({(i/len(df_us)*100):.1f}%)")

In [ ]:
# Temperature
avg_temp = df_us["Temperature(F)"].mean()
min_temp = df_us["Temperature(F)"].min()
max_temp = df_us["Temperature(F)"].max()
print(f'Temperature: \nAvg {avg_temp:.1f}°F\nMin {min_temp}°F\nMax {max_temp}°F\n')

# Visibility
avg_visibility = df_us["Visibility(mi)"].mean()
low_visibility_count = len(df_us[df_us["Visibility(mi)"] < 15])
print(f'Visibility: \nAvg {avg_visibility:.2f} mi\n')

# Wind Speed
avg_wind = df_us["Wind_Speed(mph)"].mean()
high_wind_count = len(df_us[df_us["Wind_Speed(mph)"] > 10])
print(f'Wind Speed: \nAvg {avg_wind:.2f} mph')

# SECTION 6: VISUALIZE (charts and dashboards)

In [ ]:
# Top States as per accidents chart
state_data = [df_us['State'].value_counts().head(10)]

fig1 = px.bar(
    x=state_data[0].values, 
    y=state_data[0].index,
    title='Top 10 States with Most Accidents',
    labels={'x': 'Number of Accidents', 'y': 'State'}
)

fig1.show()

In [ ]:
# Severity levels
sev_labels = {1: 'Low', 2: 'Med', 3: 'High', 4: 'Critical'}
sev_data = df_us['Severity'].value_counts().sort_index()
sev_names = [sev_labels[i] for i in sev_data.index]

fig2 = px.pie( 
    values=sev_data.values, 
    names=sev_names, 
    title='Severity of accidents',
    color_discrete_sequence=['#2ECC71', '#F39C12', '#E74C3C', '#8B0000']
)

fig2.show()

In [ ]:
# Accidents by Hour
hourly = df_us['Hour'].value_counts().sort_index()
fig3 = go.Figure(data=[
    go.Scatter(
        x=hourly.index,
        y=hourly.values,
        mode='lines+markers',
        fill='tozeroy',
        line=dict(width=2)
    )
])
fig3.update_layout(
    title='Accidents by Hour of the Day',
    xaxis_title='Hour',
    yaxis_title='Number of Accidents',
    hovermode='x unified'
)
fig3.show()

In [ ]:
# Accidents by Day of Week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily = df_us['DayOfWeek'].value_counts().reindex(day_order)

fig4 = px.bar(
    x=day_order,
    y=daily.values,
    title='Accidents by Day of Week',
    labels={'x': 'Day', 'y': 'Number of Accidents'},
    text=daily.values,
)

fig4.show()

In [ ]:
# Top Cities as per accidents chart
city_data = df_us['City'].value_counts().head(10)

fig5 = px.bar(
    x=city_data.values,
    y=city_data.index,
    title='Top 10 Cities with most accidents',
    labels={'x': 'Number of accidents', 'y': 'City'}
)

fig5.show()

In [ ]:
# Weather chart
weather_data = df_us['Weather_Condition'].value_counts().head(5)

fig6 = px.bar(
    x=weather_data.values,
    y=weather_data.index,
    title='Top 5 Weathers cused accidents',
    labels={'x': 'Number of Accidents', 'y': 'Weather'}
)

fig6.show()

In [ ]:
# duration of accidents in hours (usin histogram)
# fig7 = go.Figure()
# fig7.add_trace(go.Histogram(
#     x=df_us['Duration_Minutes'] / 60,
#     nbinsx=50))
# fig7.update_layout(
#     title='duration of accidents in hours',
#     xaxis_title='Duration (Hours)',
#     yaxis_title='Number of Accidents'
# )

# fig7.show()

In [ ]:
# duration of accidents in hours
df_us['Duration_Hours'] = df_us['Duration_Minutes'] / 60

bins = [0, 1, 2, 3, 5, 10, 24]
labels = ['0-1h', '1-2h', '2-3h', '3-5h', '5-10h', '10h+']

df_us['Duration_Group'] = pd.cut(df_us['Duration_Hours'], bins=bins, labels=labels)

duration_counts = df_us['Duration_Group'].value_counts().sort_index()
fig7 = px.bar(
    x=duration_counts.index,
    y=duration_counts.values,
    title='Accidents Duration',
    labels={'x': 'Duration(Hours)', 'y': 'Number of Accidents'}
)


fig7.show()

In [ ]:
# accidents by temperature
df_us['Temp_Range'] = pd.cut(
    df_us['Temperature(F)'],
    bins=[-30, 0, 30, 50, 70, 90, 200],
    labels=['Freezing', 'Cold', 'Cool', 'Warm', 'Hot', 'Very Hot']
)
temp_data = df_us['Temp_Range'].value_counts().sort_index()

color_map = {
    'Freezing': '#3498DB',
    'Cold': '#85C1E9',
    'Cool': '#D6EAF8',
    'Warm': '#F7DC6F',
    'Hot': '#F39C12',
    'Very Hot': "#D62713"
}

fig8 = px.bar(
    x=temp_data.index,
    y=temp_data.values,
    title='Accidents by Temperature',
    labels={'x': 'Temperature Range', 'y': 'Accidents'},
    text=temp_data.values,
    color=temp_data.index,
    color_discrete_map=color_map
)

fig8.show()

In [ ]:
# visibility chart
df_us['Visibility_Range'] = pd.cut(
    df_us['Visibility(mi)'],
    bins=[0, 1, 3, 7, 10, 100],
    labels=['Very Low', 'Low', 'Med', 'Good', 'Excellent']
)
vis_data = df_us['Visibility_Range'].value_counts().sort_index()

fig9 = px.bar(
    x=vis_data.index,
    y=vis_data.values,
    title='Accidents by Visibility',
    labels={'x': 'Visibility Level', 'y': 'Accidents'},
    text=vis_data.values,

)

fig9.show()

In [ ]:
# the accidents caused monthly
monthly = df_us['Month'].value_counts().sort_index()
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
month_labels = [month_names[i-1] for i in monthly.index]

fig10 = go.Figure()
fig10.add_trace(go.Scatter(
    x=month_labels,
    y=monthly.values,
    mode='lines+markers',
    fill='tozeroy',
))

fig10.update_layout(
    title='Accidents by Month',
    xaxis_title='Month',
    yaxis_title='Number of Accidents',
)

fig10.show()

In [ ]:
%%writefile D:\mid_project\scripts\app.py

import streamlit as st 
import pandas as pd 
import numpy as np 
import plotly.express as px 
import plotly.graph_objects as go

# Load the cleaned data
df_us = pd.read_csv('D:/mid_project/data/us_accidents_cleaned.csv')

# convert datetime columns
df_us['Start_Time'] = pd.to_datetime(df_us['Start_Time'])
df_us['Hour'] = df_us['Start_Time'].dt.hour
df_us['DayOfWeek'] = df_us['Start_Time'].dt.day_name()
df_us['Month'] = df_us['Start_Time'].dt.month

# Streamlit page configuration
st.set_page_config(page_title="US Accidents Dashboard", layout="wide")
st.title("US Accidents Dashboard")

# Sidebar filters by state and severity
st.sidebar.header("Filters")
all_states = sorted(df_us['State'].unique())
top_states = df_us['State'].value_counts().head(5).index.tolist() 
selected_states = st.sidebar.multiselect("Select States:", all_states, top_states)
selected_severity = st.sidebar.multiselect("Select Severity:", sorted(df_us['Severity'].unique()), default=sorted(df_us['Severity'].unique()))

# Filter data based on selections
filtered_df_us = df_us[(df_us['State'].isin(selected_states)) & (df_us['Severity'].isin(selected_severity))]

# KPI Metrics
col1, col2, col3, col4 = st.columns(4)
with col1:
    st.metric("Total Accidents", f"{len(filtered_df_us)}")
with col2:
    st.metric("States Covered", filtered_df_us['State'].nunique())
with col3:
    st.metric("Cities Covered", filtered_df_us['City'].nunique())
with col4:
    st.metric("Avg Duration (min)", f"{filtered_df_us['Duration_Minutes'].mean():.1f}")


# State and Severity
col1, col2 = st.columns(2)

with col1:
    state_data = filtered_df_us['State'].value_counts().head(10)
    fig_states = px.bar(x=state_data.values, y=state_data.index, 
                        title="Top 10 States", labels={'x': 'Accidents', 'y': 'State'},
                        orientation='h')
    st.plotly_chart(fig_states, use_container_width=True)

with col2:
    severity_labels = {1: 'Low', 2: 'Med', 3: 'High', 4: 'Critical'}
    sev_data = filtered_df_us['Severity'].value_counts().sort_index()
    sev_names = [severity_labels[i] for i in sev_data.index]
    fig_severity = px.pie(values=sev_data.values, names=sev_names,
                          title="Severity Distribution",
                          color_discrete_sequence=['#2ECC71', '#F39C12', '#E74C3C', '#8B0000'])
    st.plotly_chart(fig_severity, use_container_width=True)


# Hour and Day
col1, col2 = st.columns(2)

with col1:
    hours = filtered_df_us['Hour'].value_counts().sort_index()
    fig_hours = go.Figure(data=[go.Scatter(x=hours.index, y=hours.values, mode='lines+markers', fill='tozeroy')])
    fig_hours.update_layout(title="Accidents by Hour", xaxis_title="Hour", yaxis_title="Count")
    st.plotly_chart(fig_hours, use_container_width=True)

with col2:
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    day = filtered_df_us['DayOfWeek'].value_counts().reindex(day_order)
    fig_day = px.bar(x=day_order, y=day.values, title="Accidents by Day", labels={'x': 'Day', 'y': 'Accidents'})
    st.plotly_chart(fig_day, use_container_width=True)

# Top Cities and Weather
col1, col2 = st.columns(2)

with col1:
    city_data = filtered_df_us['City'].value_counts().head(5)
    fig_cities = px.bar(x=city_data.values, y=city_data.index, title="Top 5 Cities into states you selected ", labels={'x': 'Accidents', 'y': 'City'})
    st.plotly_chart(fig_cities, use_container_width=True)

with col2:
    weather_data = filtered_df_us['Weather_Condition'].value_counts().head(5)
    fig_weather = px.bar(x=weather_data.values, y=weather_data.index, title="Top 5 Weather Conditions", labels={'x': 'Accidents', 'y': 'Weather'})
    st.plotly_chart(fig_weather, use_container_width=True)


# Temperature and Visibility
col1, col2 = st.columns(2)

with col1:
    filtered_df_us['Temp_Range'] = pd.cut(filtered_df_us['Temperature(F)'],
                                       bins=[-30, 0, 30, 50, 70, 90, 200],
                                       labels=['Freezing', 'Cold', 'Cool', 'Warm', 'Hot', 'Very Hot'])
    temp_data = filtered_df_us['Temp_Range'].value_counts().sort_index()
    fig_temp = px.bar(x=temp_data.index, y=temp_data.values, title="Accidents by Temperature",
                      labels={'x': 'Temperature Range', 'y': 'Accidents'})
    st.plotly_chart(fig_temp, use_container_width=True)

with col2:
    filtered_df_us['Visibility_Range'] = pd.cut(filtered_df_us['Visibility(mi)'],
                                             bins=[0, 0.5, 1, 5, 10, 50],
                                             labels=['Very Low', 'Low', 'Med', 'Good', 'Excellent'])
    vis_data = filtered_df_us['Visibility_Range'].value_counts().sort_index()
    fig_vis = px.bar(x=vis_data.index, y=vis_data.values, title="Accidents by Visibility",
                     labels={'x': 'Visibility Level', 'y': 'Accidents'})
    st.plotly_chart(fig_vis, use_container_width=True)


# duration
col1, col2 = st.columns(2)

with col1:
    monthly = filtered_df_us['Month'].value_counts().sort_index()
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    month_labels = [month_names[i-1] for i in monthly.index]
    fig_monthly = go.Figure(data=[go.Scatter(x=month_labels, y=monthly.values, mode='lines+markers', fill='tozeroy')])
    fig_monthly.update_layout(title="Monthly", xaxis_title="Month", yaxis_title="Accidents")
    st.plotly_chart(fig_monthly, use_container_width=True)



# Data Table
st.subheader("Data")
st.dataframe(filtered_df_us[['Start_Time', 'State', 'City', 'Severity', 'Temperature(F)', 
                          'Visibility(mi)', 'Weather_Condition', 'Duration_Minutes']].head())

st.write('---')
# total records 
st.header("\nTotal Records Report")
st.write(f"- Total records in Dataset: {len(df_us)}")
st.write(f"- Total records displayed: {len(filtered_df_us)}")
st.write(f"- Date range: {df_us['Start_Time'].min().strftime('%Y-%m-%d')} to {df_us['Start_Time'].max().strftime('%Y-%m-%d')}")
st.write(f"- Unique states: {df_us['State'].nunique()}")
st.write(f"- Unique severity Levels: {df_us['Severity'].nunique()}")
st.write(f"- Unique cities: {df_us['City'].nunique()}")
st.write(f"- Unique Weather conditions: {df_us['Weather_Condition'].nunique()}")
st.write(f"- Average Duration for accidents: {filtered_df_us['Duration_Minutes'].mean():.2f} minutes")
st.write(f"- Average Visibility drivers: {filtered_df_us['Visibility(mi)'].mean():.2f} miles")
st.write(f"- Average Temperature: {filtered_df_us['Temperature(F)'].mean():.1f}°F")


# Severity statistics
st.subheader("\nSeverity Breakdown:")
severity_labels = {1: 'Low', 2: 'Med', 3: 'High', 4: 'Critical'}
severity_report = filtered_df_us['Severity'].value_counts().sort_index()
for sev_level, count in severity_report.items():
    percentage = (count / len(filtered_df_us)) * 100
    st.write(f"- {severity_labels[sev_level]}: {count} ({percentage:.2f}%)")

# Top 5 Covered States statistics as per selection
st.subheader("\nCovered States Statistics:")
state_report = filtered_df_us['State'].value_counts().nlargest()
for (state, count) in state_report.items():
    percentage = (count / len(filtered_df_us)) * 100
    st.write(f"- {state}: {count} ({percentage:.2f}%)")

# Top 5 Covered States statistics as per selection
st.subheader("\nCovered Cities Statistics:")
city_report = filtered_df_us['City'].value_counts().nlargest()
for (city, count) in city_report.items():
    percentage = (count / len(filtered_df_us)) * 100
    st.write(f"- {city}: {count} ({percentage:.2f}%)")

# Weather conditions statistics
st.subheader("Top Weather Conditions")
weather_report = filtered_df_us['Weather_Condition'].value_counts().head(5)
for weather, count in weather_report.items():
    percentage = (count / len(filtered_df_us)) * 100
    st.write(f"- {weather}: {count:,} ({percentage:.2f}%)")







In [ ]:
!streamlit run D:\mid_project\scripts\app.py